# Loadsmart — Analytics Engineer Challenge

Notebook com funções utilitárias e exportações a partir do DuckDB.

**Pré-requisitos** (já instalados no `.venv`):
```
pip install duckdb pandas paramiko
```

Execute o pipeline antes de rodar este notebook:
```bash
python scripts/ingest.py
cd dbt && dbt run
```

## 1. Setup

In [2]:
import os
import io
import smtplib
import email.mime.multipart
import email.mime.text
import email.mime.base
import email.encoders
from pathlib import Path
from dateutil.relativedelta import relativedelta

import duckdb
import pandas as pd

PROJECT_ROOT = Path(os.getcwd()).parent if Path(os.getcwd()).name == "notebooks" else Path(os.getcwd())
DUCKDB_PATH = str(PROJECT_ROOT / "data" / "loadsmart.duckdb")

print(f"DuckDB: {DUCKDB_PATH}")
print(f"Arquivo existe: {Path(DUCKDB_PATH).exists()}")


def load_smtp_config_from_env():
    host = os.environ.get("SMTP_HOST")
    user = os.environ.get("SMTP_USER")
    password = os.environ.get("SMTP_PASSWORD")
    if not all([host, user, password]):
        raise ValueError("Defina SMTP_HOST, SMTP_USER e SMTP_PASSWORD no ambiente.")
    return {
        "host": host,
        "port": int(os.environ.get("SMTP_PORT", "587")),
        "user": user,
        "password": password,
    }

DuckDB: /Users/abner.rodrigues/cases/loadsmart_case/data/loadsmart.duckdb
Arquivo existe: True


## 2. Conexão com DuckDB

In [3]:
con = duckdb.connect(DUCKDB_PATH, read_only=True)

# Confirma tabelas disponíveis
con.execute("SHOW ALL TABLES").df()

,database,schema,name,column_names,column_types,temporary
0,loadsmart,main_intermediate,int_shipments,"[LOADSMART_ID, LANE_RAW, PICKUP_CITY, PICKUP_S...","[BIGINT, VARCHAR, VARCHAR, VARCHAR, VARCHAR, V...",False
1,loadsmart,main_intermediate,int_shipments_enriched,"[loadsmart_id, lane_raw, pickup_city, pickup_s...","[BIGINT, VARCHAR, VARCHAR, VARCHAR, VARCHAR, V...",False
2,loadsmart,main_mart,dim_carrier,"[carrier_sk, carrier_name, carrier_rating, vip...","[VARCHAR, VARCHAR, DOUBLE, BOOLEAN, BIGINT]",False
3,loadsmart,main_mart,dim_date,"[DATE_SK, DATE_DAY, YEAR, QUARTER, MONTH, MONT...","[VARCHAR, TIMESTAMP, INTEGER, INTEGER, INTEGER...",False
4,loadsmart,main_mart,dim_location,"[location_sk, city, state]","[VARCHAR, VARCHAR, VARCHAR]",False
5,loadsmart,main_mart,dim_shipper,"[shipper_sk, SHIPPER_NAME]","[VARCHAR, VARCHAR]",False
6,loadsmart,main_mart,fct_shipments,"[LOADSMART_ID, carrier_sk, shipper_sk, pickup_...","[BIGINT, VARCHAR, VARCHAR, VARCHAR, VARCHAR, V...",False
7,loadsmart,main_staging,stg_shipments,"[LOADSMART_ID, LANE_RAW, PICKUP_CITY, PICKUP_S...","[BIGINT, VARCHAR, VARCHAR, VARCHAR, VARCHAR, V...",False
8,loadsmart,raw,shipments,"[loadsmart_id, lane, quote_date, book_date, so...","[BIGINT, VARCHAR, VARCHAR, VARCHAR, VARCHAR, V...",False


## 3. Função `split_lane()`

Recebe uma string de lane no formato `"City,ST -> City,ST"` e retorna um dicionário com as cidades e estados de origem e destino.

In [4]:
def split_lane(lane):
    """Parseia 'City,ST -> City,ST' em dict com pickup/delivery city e state."""
    if not lane or " -> " not in lane:
        raise ValueError(f"Lane inválida: {lane!r}")

    origin, destination = lane.split(" -> ", maxsplit=1)

    def parse_side(side):
        parts = side.split(",", maxsplit=1)
        if len(parts) != 2:
            raise ValueError(f"Lado malformado: {side!r}")
        return parts[0].strip(), parts[1].strip()

    pickup_city, pickup_state = parse_side(origin)
    delivery_city, delivery_state = parse_side(destination)

    return {
        "pickup_city": pickup_city,
        "pickup_state": pickup_state,
        "delivery_city": delivery_city,
        "delivery_state": delivery_state,
    }


# testes
cases = [
    ("Chicago,IL -> Dallas,TX",      {"pickup_city": "Chicago",  "pickup_state": "IL", "delivery_city": "Dallas",       "delivery_state": "TX"}),
    ("New York,NY -> Los Angeles,CA", {"pickup_city": "New York", "pickup_state": "NY", "delivery_city": "Los Angeles",  "delivery_state": "CA"}),
    (" Miami , FL ->  Atlanta , GA ", {"pickup_city": "Miami",    "pickup_state": "FL", "delivery_city": "Atlanta",      "delivery_state": "GA"}),
]

for lane_str, expected in cases:
    result = split_lane(lane_str)
    assert result == expected, f"FALHA: {lane_str!r} → {result}"
    print(f"OK  {lane_str!r}")

OK  'Chicago,IL -> Dallas,TX'
OK  'New York,NY -> Los Angeles,CA'
OK  ' Miami , FL ->  Atlanta , GA '


In [6]:
# Aplicando em amostra real do DuckDB
sample = con.execute("""
    SELECT LANE_RAW
    FROM main_mart.fct_shipments
    LIMIT 5
""").df()

pd.DataFrame(sample["LANE_RAW"].apply(split_lane).tolist())

,pickup_city,pickup_state,delivery_city,delivery_state
0,Hawkins,TX,Roanoke,TX
1,Hawkins,TX,Roanoke,TX
2,Plant City,FL,Woodbridge Township,NJ
3,Hawkins,TX,Roanoke,TX
4,Philadelphia,PA,Fleetwood,PA


## 4. Função `send_csv_email()`

Exporta um DataFrame como CSV em memória (sem gravar em disco) e envia como anexo por SMTP.

In [7]:
def send_csv_email(df, recipients, smtp_config, subject="Loadsmart — Export", filename="export.csv", body="Segue em anexo."):
    buffer = io.StringIO()
    df.to_csv(buffer, index=False)
    csv_bytes = buffer.getvalue().encode("utf-8")

    msg = email.mime.multipart.MIMEMultipart()
    msg["From"] = smtp_config["user"]
    msg["To"] = ", ".join(recipients)
    msg["Subject"] = subject
    msg.attach(email.mime.text.MIMEText(body, "plain"))

    attachment = email.mime.base.MIMEBase("application", "octet-stream")
    attachment.set_payload(csv_bytes)
    email.encoders.encode_base64(attachment)
    attachment.add_header("Content-Disposition", "attachment", filename=filename)
    msg.attach(attachment)

    with smtplib.SMTP(smtp_config["host"], smtp_config["port"]) as server:
        server.starttls()
        server.login(smtp_config["user"], smtp_config["password"])
        server.sendmail(smtp_config["user"], recipients, msg.as_string())

    print(f"E-mail enviado para: {', '.join(recipients)}")

In [8]:
def send_csv_email_from_path(csv_path, subject, body, recipients, smtp_config, filename=None):
    df = pd.read_csv(csv_path)
    send_csv_email(df, recipients, smtp_config, subject=subject, body=body, filename=filename or Path(csv_path).name)

## 5. Função `send_csv_sftp()`

Exporta um DataFrame como CSV e faz upload via SFTP para um servidor remoto.

In [9]:
import paramiko

def send_csv_sftp(df, remote_path, credentials):
    buffer = io.BytesIO()
    df.to_csv(io.TextIOWrapper(buffer, encoding="utf-8"), index=False)
    buffer.seek(0)

    ssh = paramiko.SSHClient()
    ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())

    connect_kwargs = {
        "hostname": credentials["host"],
        "port": credentials.get("port", 22),
        "username": credentials["username"],
    }
    if "key_path" in credentials:
        connect_kwargs["key_filename"] = os.path.expanduser(credentials["key_path"])
    else:
        connect_kwargs["password"] = credentials["password"]

    ssh.connect(**connect_kwargs)
    sftp = ssh.open_sftp()
    sftp.putfo(buffer, remote_path)
    sftp.close()
    ssh.close()

    print(f"Upload SFTP: {credentials['host']}:{remote_path}")

ModuleNotFoundError: No module named 'paramiko'

In [10]:
def send_csv_sftp_from_path(csv_path, remote_path, credentials):
    df = pd.read_csv(csv_path)
    send_csv_sftp(df, remote_path, credentials)

## 6. Exportação: Entregas do Último Mês

Consulta `main_mart.fct_shipments` filtrando pelo **último mês disponível nos dados** (não pela data atual).
O CSV exportado contém exatamente as 9 colunas especificadas no PRD.

In [11]:
# Descobre o último mês disponível nos dados (não usa date.today())
max_date_raw = con.execute("""
    SELECT MAX(DELIVERED_AT)
    FROM main_mart.fct_shipments
    WHERE LOAD_WAS_CANCELLED = false
""").fetchone()[0]

if max_date_raw is None:
    raise ValueError("Nenhuma entrega encontrada em fct_shipments.")

# Normaliza para date (DuckDB pode retornar datetime ou date)
max_date = max_date_raw.date() if hasattr(max_date_raw, 'date') else max_date_raw

first_of_last_month = max_date.replace(day=1)
first_of_next_month = first_of_last_month + relativedelta(months=1)

print(f"MAX delivered_at nos dados: {max_date}")
print(f"Último mês disponível:      {first_of_last_month.strftime('%B %Y')}")
print(f"Período filtrado:           {first_of_last_month} → {first_of_next_month} (exclusive)")

MAX delivered_at nos dados: 2025-03-15
Último mês disponível:      March 2025
Período filtrado:           2025-03-01 → 2025-04-01 (exclusive)


In [12]:
df_last_month = con.execute("""
    SELECT
        f.LOADSMART_ID,
        f.LANE_RAW,
        f.PICKUP_CITY,
        f.PICKUP_STATE,
        f.DELIVERY_CITY,
        f.DELIVERY_STATE,
        f.BOOKED_AT,
        f.PICKUP_AT,
        f.DELIVERED_AT,
        f.BOOK_PRICE,
        f.SOURCE_PRICE,
        f.pnl                 AS PNL,
        f.MILEAGE,
        f.lead_time_days      AS LEAD_TIME_DAYS,
        f.delivered_on_time   AS DELIVERED_ON_TIME,
        f.EQUIPMENT_TYPE,
        f.SOURCING_CHANNEL,
        f.LOAD_WAS_CANCELLED,
        dc.carrier_name       AS CARRIER_NAME,
        dc.carrier_rating     AS CARRIER_RATING,
        dc.vip_carrier        AS VIP_CARRIER,
        ds.shipper_name       AS SHIPPER_NAME
    FROM main_mart.fct_shipments f
    LEFT JOIN main_mart.dim_carrier dc ON f.CARRIER_SK = dc.CARRIER_SK
    LEFT JOIN main_mart.dim_shipper ds ON f.SHIPPER_SK = ds.SHIPPER_SK
    WHERE
        f.DELIVERED_AT >= CAST(? AS DATE)
        AND f.DELIVERED_AT < CAST(? AS DATE)
        AND f.LOAD_WAS_CANCELLED = false
    ORDER BY f.DELIVERED_AT
""", [str(first_of_last_month), str(first_of_next_month)]).df()

print(f"Linhas: {len(df_last_month)}")
df_last_month.head()

Linhas: 1


,LOADSMART_ID,LANE_RAW,PICKUP_CITY,PICKUP_STATE,DELIVERY_CITY,DELIVERY_STATE,BOOKED_AT,PICKUP_AT,DELIVERED_AT,BOOK_PRICE,...,MILEAGE,LEAD_TIME_DAYS,DELIVERED_ON_TIME,EQUIPMENT_TYPE,SOURCING_CHANNEL,LOAD_WAS_CANCELLED,CARRIER_NAME,CARRIER_RATING,VIP_CARRIER,SHIPPER_NAME
0,206665369,"Lodi,CA -> Pacific,WA",Lodi,CA,Pacific,WA,2024-11-27 11:05:00,2025-03-13 20:00:00,2025-03-15 13:50:00,1955.56,...,778.9,1.71,False,RFR,None,False,Carrier 86454,4.0,False,Shipper 758


In [13]:
# Resumo do mês
print("=== Resumo do Último Mês ===")
print(f"Total de entregas:    {len(df_last_month):,}")
if len(df_last_month) > 0:
    print(f"Receita total:        $ {df_last_month['BOOK_PRICE'].sum():,.2f}")
    print(f"PnL total:            $ {df_last_month['PNL'].sum():,.2f}")
    print(f"On-time rate:         {df_last_month['DELIVERED_ON_TIME'].mean():.1%}")
    print(f"Lead time médio:      {df_last_month['LEAD_TIME_DAYS'].mean():.2f} dias")
else:
    print("Nenhuma entrega no período — verifique o range de datas do dataset.")

=== Resumo do Último Mês ===
Total de entregas:    1
Receita total:        $ 1,955.56
PnL total:            $ 5.56
On-time rate:         0.0%
Lead time médio:      1.71 dias


In [14]:
COLUMNS = {
    "LOADSMART_ID": "loadsmart_id",
    "SHIPPER_NAME": "shipper_name",
    "DELIVERED_AT": "delivery_date",
    "PICKUP_CITY": "pickup_city",
    "PICKUP_STATE": "pickup_state",
    "DELIVERY_CITY": "delivery_city",
    "DELIVERY_STATE": "delivery_state",
    "BOOK_PRICE": "book_price",
    "CARRIER_NAME": "carrier_name",
}

df_export = df_last_month[list(COLUMNS.keys())].rename(columns=COLUMNS)

output_filename = f"deliveries_{first_of_last_month.strftime('%Y_%m')}.csv"
output_path = PROJECT_ROOT / "data" / "exports" / output_filename
output_path.parent.mkdir(parents=True, exist_ok=True)

df_export.to_csv(output_path, index=False)
print(f"CSV salvo: {output_path} ({len(df_export)} linhas)")
df_export.head()

CSV salvo: /Users/abner.rodrigues/cases/loadsmart_case/data/exports/deliveries_2025_03.csv (1 linhas)


,loadsmart_id,shipper_name,delivery_date,pickup_city,pickup_state,delivery_city,delivery_state,book_price,carrier_name
0,206665369,Shipper 758,2025-03-15 13:50:00,Lodi,CA,Pacific,WA,1955.56,Carrier 86454


In [15]:
# ─── Envio por E-mail — wrapper PRD (csv_path, subject, body) ──────────────
# Exporte SMTP_HOST, SMTP_USER, SMTP_PASSWORD (e opcionalmente SMTP_PORT) antes de rodar.
#
# send_csv_email_from_path(
#     csv_path=str(output_path),
#     subject=f"Deliveries — {first_of_last_month.strftime('%B %Y')}",
#     body="Segue em anexo o CSV do último mês disponível.",
#     recipients=["destinatario@example.com"],
#     smtp_config=load_smtp_config_from_env(),
# )

# ─── Envio por SFTP — wrapper PRD (csv_path, remote_path) ───────────────────
#
# send_csv_sftp_from_path(
#     csv_path=str(output_path),
#     remote_path=f"/exports/{output_filename}",
#     credentials={
#         "host": "sftp.loadsmart.com",
#         "username": "deploy",
#         "key_path": "~/.ssh/id_rsa",
#     },
# )

print("Células de envio comentadas — descomente e preencha com suas credenciais.")

Células de envio comentadas — descomente e preencha com suas credenciais.


In [16]:
con.close()
print("Conexão DuckDB fechada.")

Conexão DuckDB fechada.
